In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import re

# ---------------------------
# 1. DATA PREPROCESSING
# ---------------------------

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z?.!,]+", " ", text)
    return text

pairs = [
    ("Hello", "Hi how are you"),
    ("How are you", "I am fine"),
    ("What is your name", "I am a chatbot"),
]

pairs = [(clean_text(q), clean_text(a)) for q, a in pairs]

def tokenize(sentence):
    return sentence.split()

# Build vocab
vocab = {"<pad>":0, "<sos>":1}
for q, a in pairs:
    for word in tokenize(q) + tokenize(a):
        if word not in vocab:
            vocab[word] = len(vocab)

inv_vocab = {i:w for w,i in vocab.items()}

def sentence_to_tensor(sentence):
    return torch.tensor([vocab[word] for word in tokenize(sentence)])

# ---------------------------
# 2. MODEL
# ---------------------------

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]

        hidden = hidden.repeat(seq_len, 1, 1).permute(1, 0, 2)

        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )

        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)  # [1] -> [1,1]
        embedded = self.embedding(x)

        attn_weights = self.attention(hidden[-1].unsqueeze(1), encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)

        context = torch.bmm(attn_weights, encoder_outputs)

        lstm_input = torch.cat((embedded, context), dim=2)

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell


# ---------------------------
# 3. INITIALIZATION
# ---------------------------

vocab_size = len(vocab)

encoder = Encoder(vocab_size, 64, 128)
decoder = Decoder(vocab_size, 64, 128)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

# ---------------------------
# 4. TRAINING LOOP
# ---------------------------

for epoch in range(100):

    total_loss = 0

    for q, a in pairs:

        input_tensor = sentence_to_tensor(q).unsqueeze(0)
        target_tensor = sentence_to_tensor(a).unsqueeze(0)

        encoder_outputs, hidden, cell = encoder(input_tensor)

        decoder_input = torch.tensor([vocab["<sos>"]])

        loss = 0

        for t in range(target_tensor.shape[1]):

            output, hidden, cell = decoder(
                decoder_input, hidden, cell, encoder_outputs
            )

            target_word = target_tensor[0][t]

            loss += criterion(output, target_word.unsqueeze(0))

            decoder_input = target_word.unsqueeze(0)

        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# ---------------------------
# 5. GENERATION
# ---------------------------

def generate(sentence):
    sentence = clean_text(sentence)

    input_tensor = sentence_to_tensor(sentence).unsqueeze(0)

    encoder_outputs, hidden, cell = encoder(input_tensor)

    decoder_input = torch.tensor([vocab["<sos>"]])

    result = []

    for _ in range(10):

        output, hidden, cell = decoder(
            decoder_input, hidden, cell, encoder_outputs
        )

        top1 = output.argmax(1)

        word = inv_vocab[top1.item()]

        result.append(word)

        decoder_input = top1

    return " ".join(result)


# ---------------------------
# 6. TEST
# ---------------------------

print("\nChatbot Test:")
print("Input: how are you")
print("Output:", generate("how are you"))

Epoch 1, Loss: 30.5297
Epoch 2, Loss: 28.6154
Epoch 3, Loss: 26.8467
Epoch 4, Loss: 24.9176
Epoch 5, Loss: 22.7193
Epoch 6, Loss: 20.2123
Epoch 7, Loss: 17.4640
Epoch 8, Loss: 14.6625
Epoch 9, Loss: 11.9999
Epoch 10, Loss: 9.5624
Epoch 11, Loss: 7.4885
Epoch 12, Loss: 5.8887
Epoch 13, Loss: 4.5997
Epoch 14, Loss: 3.5751
Epoch 15, Loss: 2.8037
Epoch 16, Loss: 2.1700
Epoch 17, Loss: 1.6904
Epoch 18, Loss: 1.3123
Epoch 19, Loss: 1.0191
Epoch 20, Loss: 0.7974
Epoch 21, Loss: 0.6277
Epoch 22, Loss: 0.4980
Epoch 23, Loss: 0.3990
Epoch 24, Loss: 0.3247
Epoch 25, Loss: 0.2691
Epoch 26, Loss: 0.2274
Epoch 27, Loss: 0.1956
Epoch 28, Loss: 0.1711
Epoch 29, Loss: 0.1517
Epoch 30, Loss: 0.1362
Epoch 31, Loss: 0.1236
Epoch 32, Loss: 0.1132
Epoch 33, Loss: 0.1046
Epoch 34, Loss: 0.0972
Epoch 35, Loss: 0.0909
Epoch 36, Loss: 0.0853
Epoch 37, Loss: 0.0805
Epoch 38, Loss: 0.0761
Epoch 39, Loss: 0.0723
Epoch 40, Loss: 0.0688
Epoch 41, Loss: 0.0656
Epoch 42, Loss: 0.0627
Epoch 43, Loss: 0.0600
Epoch 44, L